In [40]:
#import and config 
import os
import time
import json
import numpy as np
import pandas as pd
import redis
import requests
from pyspark.sql import SparkSession

processed_path = "../dataset/processed"
models_path = "../Models"
outputs_path = "../Outputs"

os.makedirs(outputs_path, exist_ok=True)

print("Infrastructure integration environment ready.")

Infrastructure integration environment ready.


In [41]:
# Load Two-Tower Models and Weights

user_ids = retrieval_interactions["user_id"].astype(str).unique().tolist()

movie_ids_vocab = np.load(
    os.path.join(models_path, "movie_ids.npy")
).astype(str).tolist()

user_model = UserModel(user_ids)
movie_model = MovieModel(movie_ids_vocab)

# Build models
user_model(tf.constant([user_ids[0]]))
movie_model(tf.constant([movie_ids_vocab[0]]))

# Load trained weights
user_model.load_weights(
    os.path.join(models_path, "user_model.weights.h5")
)

movie_model.load_weights(
    os.path.join(models_path, "movie_model.weights.h5")
)

print("User model loaded:", len(user_ids))
print("Movie model loaded:", len(movie_ids_vocab))
print("Both trained models loaded successfully.")

User model loaded: 6038
Movie model loaded: 3456
Both trained models loaded successfully.


In [42]:
#testing the redis connection 
redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis connection:", redis_client.ping())

Redis connection: True


In [43]:
# Initialize and Test Spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RecommendationEngine")
    .master("local[*]")
    .getOrCreate()
)

ratings_pd = pd.read_parquet(
    os.path.join(processed_path, "interactions.parquet")
)

ratings_pd["timestamp"] = pd.to_datetime(
    ratings_pd["timestamp"]
).astype("datetime64[us]")

spark_compatible_path = os.path.join(
    processed_path, "interactions_spark_compatible.parquet"
)

ratings_pd.to_parquet(
    spark_compatible_path,
    index=False,
    engine="pyarrow"
)

ratings_df = spark.read.parquet(spark_compatible_path)

row_count = ratings_df.count()

print("Spark version:", spark.version)
print("Ratings rows:", row_count)
print("Dataset shape:", row_count, len(ratings_df.columns))

Spark version: 3.5.9
Ratings rows: 1000209
Dataset shape: 1000209 27


In [12]:
#testing the spark data 
spark_user_stats = (
    ratings_df
    .groupBy("user_id")
    .agg(
        {"rating": "count"}
    )
    .withColumnRenamed("count(rating)", "total_ratings")
)

print("Users processed:", spark_user_stats.count())

spark_user_stats.show(5)

Users processed: 6040
+-------+-------------+
|user_id|total_ratings|
+-------+-------------+
|   5925|          197|
|   5645|          270|
|   5325|           48|
|   4937|          214|
|   4821|          234|
+-------+-------------+
only showing top 5 rows



In [13]:
#saving the processed data 
spark_stats_path = os.path.join(
    outputs_path, "spark_user_statistics.parquet"
)

spark_user_stats.toPandas().to_parquet(
    spark_stats_path,
    index=False
)

print("Spark statistics saved:", spark_stats_path)

Spark statistics saved: ../Outputs\spark_user_statistics.parquet


In [44]:
#dag locations 
airflow_dag_path = r"D:\Pradeep Pandey\Nexvia Technology  Intership\airflow\dags"

print("Airflow DAG folder exists:", os.path.exists(airflow_dag_path))

Airflow DAG folder exists: True


In [15]:
#creating the airflow dag file
airflow_dag_path = r"D:\Pradeep Pandey\Nexvia Technology  Intership\airflow\dags"
dag_code = '''
import os
import json
from datetime import datetime

from airflow import DAG
from airflow.operators.python import PythonOperator

PROJECT_PATH = "/opt/airflow/recommendation_project"
OUTPUT_PATH = os.path.join(PROJECT_PATH, "Outputs")


def validate_data():
    files = [
        "interactions.parquet",
        "retrieval_interactions.parquet",
        "faiss_recommendations.parquet",
        "ranked_recommendations.parquet"
    ]

    missing = [
        f for f in files
        if not os.path.exists(
            os.path.join(PROJECT_PATH, "dataset", "processed", f)
        )
    ]

    if missing:
        raise FileNotFoundError(f"Missing data files: {missing}")

    print("All recommendation datasets validated.")


def validate_models():
    files = [
        "movie_faiss.index",
        "movie_ids.npy",
        "user_model.weights.h5",
        "movie_model.weights.h5",
        "ranking_model.weights.h5"
    ]

    missing = [
        f for f in files
        if not os.path.exists(os.path.join(PROJECT_PATH, "Models", f))
    ]

    if missing:
        raise FileNotFoundError(f"Missing model files: {missing}")

    print("All model artifacts validated.")


def save_pipeline_status():
    status = {
        "pipeline": "Recommendation Engine",
        "status": "success",
        "completed_at": datetime.now().isoformat()
    }

    os.makedirs(OUTPUT_PATH, exist_ok=True)

    with open(
        os.path.join(OUTPUT_PATH, "airflow_pipeline_status.json"),
        "w"
    ) as file:
        json.dump(status, file, indent=4)

    print("Pipeline status saved.")


with DAG(
    dag_id="recommendation_engine_pipeline",
    start_date=datetime(2026, 11, 22),
    schedule=None,
    catchup=False,
    tags=["recommendation", "nexvia"]
) as dag:

    data_validation = PythonOperator(
        task_id="validate_recommendation_data",
        python_callable=validate_data
    )

    model_validation = PythonOperator(
        task_id="validate_model_artifacts",
        python_callable=validate_models
    )

    save_status = PythonOperator(
        task_id="save_pipeline_status",
        python_callable=save_pipeline_status
    )

    [data_validation, model_validation] >> save_status
'''

dag_file = os.path.join(
    airflow_dag_path,
    "recommendation_engine_pipeline.py"
)

with open(dag_file, "w", encoding="utf-8") as file:
    file.write(dag_code)

print("DAG created:", dag_file)

DAG created: D:\Pradeep Pandey\Nexvia Technology  Intership\airflow\dags\recommendation_engine_pipeline.py


In [45]:
#checking the triton server
response = requests.get(
    "http://localhost:8000/v2/health/ready",
    timeout=10
)

print("Triton status:", response.status_code)
print("Server ready:", response.status_code == 200)

Triton status: 200
Server ready: True


In [46]:
#checking the loaded models 
response = requests.post(
    "http://localhost:8000/v2/repository/index",
    json={},
    timeout=10
)

print("Response:", response.status_code)
print("Model repository:", response.json())

Response: 200
Model repository: [{'name': 'ranking_model', 'version': '1', 'state': 'READY'}]


In [47]:
#relaoding the saved trained model
import tf_keras as keras

ranking_model = keras.Sequential([
    keras.layers.Input(shape=(129,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

ranking_model.load_weights(
    os.path.join(models_path, "ranking_model.weights.h5")
)

ranking_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 128)               16640     
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_4 (Dense)             (None, 64)                8256      
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 24961 (97.50 KB)
Trainable params: 24961 (97.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [21]:
#exporting the ranking models 
import tensorflow as tf
import tf2onnx
input_signature = [
    tf.TensorSpec((None, 129), tf.float32, name="input")
]

inference_function = tf.function(
    lambda x: ranking_model(x, training=False),
    input_signature=input_signature
)

onnx_path = os.path.join(models_path, "ranking_model.onnx")

model_proto, _ = tf2onnx.convert.from_function(
    inference_function,
    input_signature=input_signature,
    opset=13,
    output_path=onnx_path
)

print("ONNX model saved:", os.path.exists(onnx_path))
print("Model size:", round(os.path.getsize(onnx_path) / 1024, 2), "KB")



ONNX model saved: True
Model size: 99.03 KB


In [48]:
#verifying the onnx model 
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print("ONNX model validation: Passed")
print("Input:", onnx_model.graph.input[0].name)
print("Output:", onnx_model.graph.output[0].name)

ONNX model validation: Passed
Input: input
Output: Identity:0


In [49]:
#preparing the triton model repo
triton_model_path = os.path.join(
    "..", "model_repository", "ranking_model", "1"
)

os.makedirs(triton_model_path, exist_ok=True)

import shutil

shutil.copy2(
    onnx_path,
    os.path.join(triton_model_path, "model.onnx")
)

print("Model copied:", os.path.exists(
    os.path.join(triton_model_path, "model.onnx")
))

Model copied: True


In [26]:
#creating the triton model config 
config_path = os.path.join(
    "..", "model_repository", "ranking_model", "config.pbtxt"
)

config = """
name: "ranking_model"
platform: "onnxruntime_onnx"
max_batch_size: 1024

input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [129]
  }
]

output [
  {
    name: "Identity:0"
    data_type: TYPE_FP32
    dims: [1]
  }
]

instance_group [
  {
    count: 1
    kind: KIND_CPU
  }
]
"""

with open(config_path, "w") as file:
    file.write(config)

print("Triton configuration created:", os.path.exists(config_path))

Triton configuration created: True


In [50]:
#verifying the triton model loading 
import time

time.sleep(5)

response = requests.post(
    "http://localhost:8000/v2/repository/index",
    json={},
    timeout=10
)

print("Triton models:", response.json())

Triton models: [{'name': 'ranking_model', 'version': '1', 'state': 'READY'}]


In [28]:
#Load Recommendation Data
retrieval_interactions = pd.read_parquet(
    os.path.join(processed_path, "retrieval_interactions.parquet")
)

movie_features = pd.read_parquet(
    os.path.join(processed_path, "movie_features.parquet")
)

print("Retrieval interactions:", retrieval_interactions.shape)
print("Movie features:", movie_features.shape)

Retrieval interactions: (575281, 3)
Movie features: (3883, 20)


In [31]:
#Load Two-Tower Models and Weights
movie_ids_vocab = np.load(
    os.path.join(models_path, "movie_ids.npy")
).astype(str).tolist()

movie_model = MovieModel(movie_ids_vocab)

movie_model(tf.constant([movie_ids_vocab[0]]))

movie_model.load_weights(
    os.path.join(models_path, "movie_model.weights.h5")
)

print("Movie tower weights loaded successfully.")
print("Movies:", len(movie_ids_vocab))
print(
    "Movie embedding:",
    movie_model.movie_embedding.embeddings.shape
)


Movie tower weights loaded successfully.
Movies: 3456
Movie embedding: (3457, 64)


In [32]:
#loading the candidate recommendation
faiss_recommendations = pd.read_parquet(
    os.path.join(processed_path, "faiss_recommendations.parquet")
)

user_candidates = faiss_recommendations[
    faiss_recommendations["user_id"].astype(str) == "1"
].copy()

print("Candidates found:", len(user_candidates))
display(user_candidates.head())

Candidates found: 10


,user_id,movie_id,retrieval_score,rank
0,1,914,1.195011,1
1,1,1947,1.042930,2
2,1,2080,1.023007,3
3,1,920,0.998976,4
4,1,595,0.987576,5


In [35]:
#generating the real ranking features user_id = "1"

user_id = "1"

candidate_movie_ids = user_candidates["movie_id"].astype(str).tolist()

user_embedding = user_model(
    tf.constant([user_id])
).numpy()

movie_embeddings = movie_model(
    tf.constant(candidate_movie_ids)
).numpy()

user_embeddings = np.repeat(
    user_embedding, len(candidate_movie_ids), axis=0
)

similarity = np.sum(
    user_embeddings * movie_embeddings,
    axis=1,
    keepdims=True
)

ranking_features = np.concatenate(
    [user_embeddings, movie_embeddings, similarity],
    axis=1
).astype(np.float32)

print("Ranking features shape:", ranking_features.shape)

Ranking features shape: (10, 129)


In [36]:
#comparing the tensorflow and triton prediction
tf_scores = ranking_model.predict(
    ranking_features, verbose=0
).flatten()

payload = {
    "inputs": [{
        "name": "input",
        "shape": list(ranking_features.shape),
        "datatype": "FP32",
        "data": ranking_features.tolist()
    }],
    "outputs": [{"name": "Identity:0"}]
}

response = requests.post(
    "http://localhost:8000/v2/models/ranking_model/infer",
    json=payload,
    timeout=30
)

response.raise_for_status()

triton_scores = np.array(
    response.json()["outputs"][0]["data"]
)

max_difference = np.max(
    np.abs(tf_scores - triton_scores)
)

print("TensorFlow scores:", tf_scores[:5])
print("Triton scores:", triton_scores[:5])
print("Maximum difference:", max_difference)

TensorFlow scores: [0.9098788  0.90131325 0.82548124 0.91670984 0.71798825]
Triton scores: [0.90987885 0.90131319 0.82548124 0.9167099  0.71798825]
Maximum difference: 5.960464477539063e-08


In [37]:
#saving the triton inference results 
comparison_df = pd.DataFrame({
    "movie_id": candidate_movie_ids,
    "tensorflow_score": tf_scores,
    "triton_score": triton_scores
})

comparison_path = os.path.join(
    outputs_path, "triton_inference_comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

print("Comparison saved:", comparison_path)
print("Maximum difference:", max_difference)
display(comparison_df.head())

Comparison saved: ../Outputs\triton_inference_comparison.csv
Maximum difference: 5.960464477539063e-08


,movie_id,tensorflow_score,triton_score
0,914,0.909879,0.909879
1,1947,0.901313,0.901313
2,2080,0.825481,0.825481
3,920,0.916710,0.916710
4,595,0.717988,0.717988


In [52]:
#cache recommendation in redis 
recommendation_results = pd.DataFrame({
    "movie_id": candidate_movie_ids,
    "ranking_score": triton_scores
}).sort_values("ranking_score", ascending=False).head(10)

cache_key = f"recommendations:user:{user_id}"

redis_client.set(
    cache_key,
    recommendation_results.to_json(orient="records"),
    ex=3600
)

print("Cache saved:", redis_client.exists(cache_key))
print("Cached recommendations:", len(recommendation_results))

Cache saved: 1
Cached recommendations: 10


In [51]:
#retrive cached recommendation 
cached_recommendations = redis_client.get(cache_key)

cached_df = pd.DataFrame(
    json.loads(cached_recommendations)
)

print("Cached recommendations retrieved:", len(cached_df))
display(cached_df)

Cached recommendations retrieved: 10


,movie_id,ranking_score
0,899,0.925841
1,920,0.916710
2,914,0.909879
3,1947,0.901313
4,17,0.885400
5,953,0.873144
6,2565,0.836157
7,2080,0.825481
8,2081,0.746263
9,595,0.717988
